# Phase 1 — SimCLR reference encoder (Kaggle)

Trains the Phase-1 SimCLR reference encoder (2 seeds) + the supervised-reference ceiling on Shapes3D, then runs the encoder-quality gate (SimCLR vs random-encoder vs supervised).

**Before running** (right sidebar → Settings):
- **Accelerator → GPU P100** (single GPU; the code is single-GPU, so T4 x2 gives no benefit).
- **Internet → On** (needed to clone the repo, `pip install`, and download the dataset).

Everything lives on Kaggle's local SSD (`/kaggle/working`), which is fast and persists for the session. To keep checkpoints across sessions, **Save Version** (commit) — outputs in `/kaggle/working` are stored with the version. Training checkpoints every epoch and auto-resumes from `last_ckpt.pt`: if a cell stops partway, just re-run it.

In [ ]:
!nvidia-smi

## 1. Clone the repo

Cloned onto local SSD (`/kaggle/working`). Run cells below `cd` in by absolute path so they still work after a kernel restart (which clears the `%cd` but leaves files on disk).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 2. Install dependencies

Install the package without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Settings > Accelerator > GPU")

## 3. Download Shapes3D to local SSD

Downloads to `data/raw/` on local disk. Training reads the whole split into RAM once at startup, so the dataset must be on fast local storage (not a network mount).

Re-downloads each fresh session (~255 MB, under a minute). To skip that, upload `3dshapes.h5` once as a Kaggle Dataset and attach it under `/kaggle/input/...`, then pass `--set data.path=/kaggle/input/<name>/3dshapes.h5` to the run cells.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download

## 4. Calibrate runtime (optional, one epoch)

Run one epoch, note the wall time, multiply by `epochs` (100) for a full-run estimate. This writes the epoch-0 checkpoint, so the full run below resumes from epoch 1 — no wasted compute. Expect well under 1 s/iter; if you still see tens of s/iter, the data isn't on local disk.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml --set run.seed=0 run.epochs=1

## 5. Train the reference encoder — seed 0

Checkpoints every epoch. If this cell stops, re-run it to resume from the last saved epoch.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml --set run.seed=0

## 6. Train the reference encoder — seed 1 (seed-to-seed reproducibility check)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml --set run.seed=1

## 7. Train the supervised-reference encoder (recoverability ceiling)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.supervised \
    --config configs/run/supervised.yaml --set run.seed=0

## 8. Run the encoder-quality gate

Per-factor normalized recoverability for SimCLR vs random-encoder vs supervised-reference → PASS/FAIL against the Phase-1 thresholds.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

## 9. Print the gate report

In [ ]:
import json
print(json.dumps(json.load(open("results/quality_gate/reference.json")), indent=2))